In [ ]:
import pandas as pd
import numpy as np
import sqlalchemy as sa
import pyodbc

from sqlalchemy.engine import URL
connection_string = "DWN_name"
connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})

from sqlalchemy import create_engine
engine = create_engine(connection_url)

In [ ]:
Ottok = '''
text of sql query
'''

In [ ]:
with engine.begin() as conn:
    Ottok = pd.read_sql_query(Ottok, conn)

In [ ]:
engine.dispose()

In [ ]:
Ottok.info ()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2263353 entries, 0 to 2263352
Data columns (total 25 columns):
 #   Column                                      Dtype         
---  ------                                      -----         
 0   SAP_GPART                                   object        
 1   Дата подключения откл тарифа                datetime64[ns]
 2   Дата отключения откл тарифа                 datetime64[ns]
 3   Название откл тарифа                        object        
 4   Название откл услуги                        object        
 5   Признак акционности откл тарифа             int64         
 6   Дата регистрации                            datetime64[ns]
 7   Название спутника                           object        
 8   Парк оборудования                           object        
 9   Модель оборудования                         object        
 10  Название организации дилеров                object        
 11  Источник сигнала                            object

In [ ]:
def get_column_sizes (df):
    results = []

    for column in df.columns:
        column_size = df[column].memory_usage (deep=True)
        column_size_kb = column_size / 1024

        dtype = df[column].dtype
        empty_strings = round (df[column].isnull().sum()/len (df[column]), 2)

        results.append ({
            'Название признака': column,
            'Тип данных': dtype,
            'Размер': round (column_size_kb, 2),
            'Пустые значения': empty_strings
        })

    return results

results = get_column_sizes (Ottok)
for item in results:
    print (f'Название признака: {item['Название признака']:60} | Размер: {item['Размер']:9.2f} Кб | Пустые значения: {item['Пустые значения']} %')

Название признака: SAP_GPART                                                    | Размер: 276648.94 Кб | Пустые значения: 0.0 %
Название признака: Дата подключения откл тарифа                                 | Размер:  37511.83 Кб | Пустые значения: 0.0 %
Название признака: Дата отключения откл тарифа                                  | Размер:  37511.83 Кб | Пустые значения: 0.0 %
Название признака: Дата подкл след тарифа                                       | Размер:  37511.83 Кб | Пустые значения: 0.43 %
Название признака: Название откл тарифа                                         | Размер: 443692.78 Кб | Пустые значения: 0.0 %
Название признака: Название откл услуги                                         | Размер: 385005.15 Кб | Пустые значения: 0.0 %
Название признака: Длительность откл тарифа                                     | Размер:  37511.83 Кб | Пустые значения: 0.0 %
Название признака: Признак акционности откл тарифа                              | Размер:  37511.83 Кб 

In [ ]:
def find_binary_columns (df):
    binary_columns = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            unique_vals =  df[col].unique()
            if set (unique_vals).issubset({0, 1}):
                binary_columns.append(col)

    return binary_columns

def find_float_binary_columns (df):
    float_binary_columns = []
    for col in df.columns:
        if df[col].dtype == 'float64':
            non_nan_vals = df[col].dropna().unique()
            if len (non_nan_vals) == 1 and np.isclose (non_nan_vals[0], 1.0):
                float_binary_columns.append (col)
            elif len (non_nan_vals) == 2 and set (non_nan_vals).issubset({0.0, 1.0}):
                float_binary_columns.append (col)

    return float_binary_columns

binary_cols = find_binary_columns(Ottok)

float_binary_cols = find_float_binary_columns (Ottok)

all_binary_cols = binary_cols + float_binary_cols
print (f'Все бинарные признаки для преобразования: {all_binary_cols}')

for col in all_binary_cols:
    if col in float_binary_cols:
        Ottok[col] = Ottok[col].fillna (0).astype (bool)
    else:
        Ottok[col] = Ottok[col].astype (bool)

Все бинарные признаки для преобразования: ['Признак акционности откл тарифа', 'Признак платности откл тарифа', 'Признак кредита откл тарифа', 'Признак активности откл тарифа', 'Признак активного кредита', 'Вай-фай на оборудовании', 'Интернет на оборудовании']


In [ ]:
Ottok.info ()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4801498 entries, 0 to 4801497
Data columns (total 41 columns):
 #   Column                                                        Dtype         
---  ------                                                        -----         
 0   SAP_GPART                                                     object        
 1   Дата подключения откл тарифа                                  datetime64[ns]
 2   Дата отключения откл тарифа                                   datetime64[ns]
 3   Дата подкл след тарифа                                        datetime64[ns]
 4   Название откл тарифа                                          object        
 5   Название откл услуги                                          object        
 6   Длительность откл тарифа                                      int64         
 7   Признак акционности откл тарифа                               bool          
 8   Признак платности откл тарифа                                 

In [31]:
#Здесь преобразуем столбец с телеофнами, если нет тф, то 0, если есть, то 1

import re

def create_phone_prescence_column(df, phone_column):
    
    def is_valid_russian_phone(phone):

        if pd.isna(phone):
            return False
        
        phone_str = str(phone).strip()
        
        # Базовые проверки
        if phone_str == '':
            return False
        
        # Удаляем все нецифровые символы
        digits_only = re.sub(r'\D', '', phone_str)
        
        # Проверка длины (российские мобильные обычно 10-11 цифр)
        if len(digits_only) not in [10, 11]:
            return False
        
        # Проверка кодов российских мобильных операторов
        if len(digits_only) == 10:
            # Формат без 7/8: 9XX XXX-XX-XX
            if not digits_only.startswith('9'):
                return False
        elif len(digits_only) == 11:
            # Формат с 7/8: 79XX XXX-XX-XX или 89XX XXX-XX-XX
            if not (digits_only.startswith('79') or digits_only.startswith('89')):
                return False
        
        # Получаем последние 10 цифр (основную часть номера)
        main_digits = digits_only[-10:]
        
        # Проверка на одинаковые цифры
        if len(set(main_digits)) == 1:
            return False
        
        # Проверка на последовательности
        sequential_patterns = [
            '1234567890',  # возрастающая последовательность
            '9876543210',  # убывающая последовательность  
            '0123456789',  # с нуля
            '123456789',   # короткие последовательности
            '987654321',
            '234567890',
            '876543210',
            '12345678',
            '87654321',
            '1234567',
            '7654321',
            '123456',
            '654321',
            '12345',
            '54321',
            '1234',
            '4321',
            '123',
            '321',
            '012345678',
            '876543210',
        ]
        
        for pattern in sequential_patterns:
            if pattern in main_digits:
                return False
        
        # Проверка на зеркальные последовательности
        mirror_patterns = [
            '1234554321',  # 12345 + 54321
            '12344321',    # 1234 + 4321
            '123321',      # 123 + 321
            '1221',        # 12 + 21
            '111222111',   # 111 + 222 + 111
        ]
        
        for pattern in mirror_patterns:
            if pattern in main_digits:
                return False
        
        # Проверка на повторяющиеся группы
        repeating_patterns = [
            r'^(\d)\1{9}$',           # одна цифра повторяется 10 раз
            r'^(\d{2})\1{4}$',        # две цифры повторяются 5 раз
            r'^(\d{3})\1{3}$',        # три цифры повторяются 3 раза + 1
            r'^(\d{4})\1{2}$',        # четыре цифры повторяются 2 раза + 2
            r'^(\d{5})\1{1}$',        # пять цифр повторяются 2 раза
            r'^12341234',             # повтор 1234
            r'^12312312',             # повтор 123
            r'^12121212',             # повтор 12
        ]
        
        for pattern in repeating_patterns:
            if re.match(pattern, main_digits):
                return False
        
        # Проверка на популярные тестовые номера
        test_numbers = [
            '9991112233',  # случайный тестовый
            '1112223344',  # последовательные группы
            '7778889990',  # повторяющиеся 7,8,9
            '9999999999',  # все девятки
            '8888888888',  # все восьмёрки
            '7777777777',  # все семёрки
            '6666666666',  # все шестерки
            '5555555555',  # все пятерки
            '4444444444',  # все четверки
            '3333333333',  # все тройки
            '2222222222',  # все двойки
            '1111111111',  # все единицы
            '0000000000',  # все нули
        ]
        
        if main_digits in test_numbers:
            return False
        
        # Проверка на номера с повторяющимися парами/тройками
        if (re.search(r'(\d{2})\1\1\1\1', main_digits) or  # 5 одинаковых пар
            re.search(r'(\d{3})\1\1\1', main_digits) or    # 4 одинаковых тройки
            re.search(r'(\d{4})\1\1', main_digits) or      # 3 одинаковых четверки
            re.search(r'(\d{5})\1', main_digits)):         # 2 одинаковых пятерки
            return False
        
        # Проверка на чередующиеся цифры
        alternating_patterns = [
            r'^(01){5}',      # 0101010101
            r'^(12){5}',      # 1212121212  
            r'^(23){5}',      # 2323232323
            r'^(34){5}',      # 3434343434
            r'^(45){5}',      # 4545454545
            r'^(56){5}',      # 5656565656
            r'^(67){5}',      # 6767676767
            r'^(78){5}',      # 7878787878
            r'^(89){5}',      # 8989898989
            r'^(90){5}',      # 9090909090
            r'^(09){5}',      # 0909090909
        ]
        
        for pattern in alternating_patterns:
            if re.match(pattern, main_digits):
                return False
        
        # Проверка на симметричные номера
        symmetric_patterns = [
            '1223332211',  # симметричный паттерн
            '1112221111',  # симметричные группы
            '1234554321',  # полная симметрия
            '1234432111',  # частичная симметрия
        ]
        
        for pattern in symmetric_patterns:
            if pattern in main_digits:
                return False
        
        # Дополнительная проверка: если номер содержит более 80% одинаковых цифр
        digit_counts = {digit: main_digits.count(digit) for digit in set(main_digits)}
        max_count = max(digit_counts.values())
        if max_count >= 8:
            return False
        
        return True
    
    df['Указан номер телефона'] = df[phone_column].apply(is_valid_russian_phone)
    
    return df

Ottok = create_phone_prescence_column(Ottok, 'наличие мобильного телефона')

In [32]:
#Здесь проверяем был ли у абонента обмен оборудования по трейд-ин

def create_change_equipment_column (df, equipment_column):
    df['Обменивал оборудование'] = df[equipment_column].notna() & (df[equipment_column].astype(str).str.strip() != '').astype(bool)

    return df

Ottok = create_change_equipment_column (Ottok, 'Дата обмена оборудования')

In [33]:
#Здесь проверяем обращался ли абонент в КЦ за последние 1 месяца до даьты отключения

def subscriber_contacted_column (df, contacted_column):
    df['Обращался в КЦ'] = df[contacted_column].notna() & (df[contacted_column].astype(str).str.strip() != '').astype(bool)

    return df

Ottok = subscriber_contacted_column (Ottok, 'Дата и время обращения в КЦ')

In [34]:
#Здесь извлекаем из даты номер недели, номер месяца, номер месяца, день недели, номер дня месяца

def extract_datetime_features(df, datetime_columns):

    for col in datetime_columns:
        # Номер недели (ISO неделя)
        df[f'{col}_Номер недели'] = df[col].dt.isocalendar().week
        
        # Номер года
        df[f'{col}_Год'] = df[col].dt.year
        
        # Номер месяца
        df[f'{col}_Месяц'] = df[col].dt.month
        
        # Номер дня месяца
        df[f'{col}_День месяца'] = df[col].dt.day
        
        # День недели (понедельник=0, воскресенье=6)
        df[f'{col}_День недели'] = df[col].dt.dayofweek
    
    return df

# Использование функции
datetime_columns = ['Дата отключения откл тарифа']
Ottok = extract_datetime_features(Ottok, datetime_columns)

In [35]:
#Здесь укрупняется ИП

def replace_ip_in_dealer_name(df):

    df['Название организации дилеров'] = df['Название организации дилеров'].apply(
        lambda x: 'ИП' if 'ИП' in str(x) else x
    )
    
    return df

Ottok = replace_ip_in_dealer_name (Ottok)

In [36]:
#Здесь укрупняется ФЛ

def replace_ip_in_dealer_name(df):

    df['Название организации дилеров'] = df['Название организации дилеров'].apply(
        lambda x: 'ФЛ' if 'ФЛ' in str(x) else x
    )
    
    return df

Ottok = replace_ip_in_dealer_name (Ottok)

In [37]:
#проверка на последнюю неделю
#Ottok [(Ottok['Дата отключения откл тарифа_Номер недели']==44) & (Ottok['Признак акционности след тарифа']==True)][['Дата подключения откл тарифа', 'Дата отключения откл тарифа', 'Название откл тарифа', 'Название откл услуги', 'Длительность откл тарифа', 'Признак акционности откл тарифа', 'Признак платности откл тарифа', 'Признак кредита откл тарифа', 'Признак активности откл тарифа', 'Дата подкл след тарифа', 'Дата откл след тарифа', 'Название след тарифа', 'Название след услуги', 'Длительность след тарифа', 'Признак акционности след тарифа', 'Признак платности след тарифа', 'Признак кредита след тарифа', 'Признак активности след тарифа']].head(10)

In [38]:
def calculate_customer_lifetime (df, registration_col, churn_col, new_col='Время жизни в месяцах'):
    df['lifetime_days'] = (df[churn_col] - df[registration_col]).dt.days
    df[new_col] = (df['lifetime_days']/30).round().astype(int)
    df = df.drop ('lifetime_days', axis=1)

    return df

def categorized_lifetime (df, lifetime_col, category_col = 'Категория по времени жизни'):
    conditions = [
        df[lifetime_col] < 12,
        (df[lifetime_col] > 12) & (df[lifetime_col] <= 24),
        (df[lifetime_col] > 24) & (df[lifetime_col] <= 36),
        (df[lifetime_col] > 36) & (df[lifetime_col] <= 48),
        (df[lifetime_col] > 48) & (df[lifetime_col] <= 60),
        (df[lifetime_col] > 60) & (df[lifetime_col] <= 120),
        df[lifetime_col] > 120
    ]

    categories = ['до года',
        '1-2 года', '2-3 года', '3-4 года', '4-5 лет', '5-10 лет', '10+ лет'
    ]

    df[category_col] = np.select (conditions, categories, default='нет данных')

    return df

def process_customer_lifetime (df):
    registration_col='Дата регистрации'
    churn_col='Дата отключения откл тарифа'

    df=calculate_customer_lifetime (df, registration_col, churn_col, 'Время жизни в месяцах')
    df=categorized_lifetime (df, 'Время жизни в месяцах', 'Категория по времени жизни')

    return df

Ottok = process_customer_lifetime (Ottok)

In [39]:
def create_service_flags (df, service_date_columns, churn_date_column):
    for service_col in service_date_columns:
        service_name = service_col.replace('_Дата окончания последней услуги', '')

        df[f'{service_name}_активна_на_момент_отключения'] = (
            (df[service_col].notna()) &
            (df[service_col] > df[churn_date_column])
        ).astype(bool)

    return df

service_column = ['Дата окончания последней услуги Детский', 'Дата окончания последней услуги Ночной',
                  'Дата окончания последней услуги Матч! Футбол/Весь Футбол', 'Дата окончания последней услуги Ultra',
                  'Дата окончания последней услуги Добавь Кино/Добавь Кино и 4К', 'Дата окончания последней услуги Матч Премьер',
                  'Дата окончания последней услуги START', 'Дата окончания последней услуги Premier',
                  'Дата окончания последней услуги Amediateka', 'Дата окончания последней услуги Мультирум'
                 ]
churn_date_column = 'Дата отключения откл тарифа'

Ottok = create_service_flags (Ottok, service_column, churn_date_column)

In [40]:
Ottok.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4801498 entries, 0 to 4801497
Data columns (total 61 columns):
 #   Column                                                                                     Dtype         
---  ------                                                                                     -----         
 0   SAP_GPART                                                                                  object        
 1   Дата подключения откл тарифа                                                               datetime64[ns]
 2   Дата отключения откл тарифа                                                                datetime64[ns]
 3   Дата подкл след тарифа                                                                     datetime64[ns]
 4   Название откл тарифа                                                                       object        
 5   Название откл услуги                                                                       object        

In [41]:
def create_final_dataframe (df, columns_exclude):
    all_columns = df.columns.tolist()

    final_columns = [col for col in all_columns if col not in columns_exclude]

    final_df = df[final_columns]

    return final_df


delete_columns = ['наличие мобильного телефона', 'Дата обмена оборудования', 'EMID', 'SAP_GPART', 'Статус ОУ на сейчас',
'Дата отключения откл тарифа', 'Дата подкл след тарифа', 'Дата откл след тарифа', 'Дата и время обращения в КЦ',
'Тариф последней услуги на момент обращения', 'Дата регистрации', 'Дата отключения откл тарифа', 'Время жизни в месяцах',
'Дата окончания последней услуги Детский', 'Дата окончания последней услуги Ночной', 'Дата отключения откл тарифа',
'Дата окончания последней услуги Матч! Футбол/Весь Футбол', 'Дата окончания последней услуги Ultra', 'Последняя услуга на момент обращения',
'Дата окончания последней услуги Добавь Кино/Добавь Кино и 4К', 'Дата окончания последней услуги Матч Премьер',
'Дата окончания последней услуги START', 'Дата окончания последней услуги Premier', 'Дата подключения откл тарифа',
'Дата окончания последней услуги Amediateka', 'Дата окончания последней услуги Мультирум',
'Дата подключения откл тарифа_Номер месяца', 'Дата подключения откл тарифа_День недели', 'Дата подключения откл тарифа_Номер дня месяца',
'Дата подключения откл тарифа_Номер недели', 'Дата отключения откл тарифа_Номер месяца', 'Дата отключения откл тарифа_День недели',
'Дата отключения откл тарифа_Номер дня месяца', 'Дата подкл след тарифа_Номер недели', 'Дата подкл след тарифа_Номер месяца',
'Дата подкл след тарифа_День недели', 'Дата подкл след тарифа_Номер дня месяца', 'Дата откл след тарифа_Номер недели',
'Дата откл след тарифа_Номер месяца', 'Дата откл след тарифа_День недели', 'Дата откл след тарифа_Номер дня месяца',
'Дата подключения откл тарифа_Номер месяца ', 'Дата подключения откл тарифа_День недели ', 'Дата отключения откл тарифа_Год '
'Дата отключения откл тарифа_Год', 'Дата отключения откл тарифа_Месяц', 'Дата отключения откл тарифа_День месяца',
'Дата отключения откл тарифа_День недели', 'Дата окончания последней услуги Детский_активна_на_момент_отключения',
'Дата окончания последней услуги Ночной_активна_на_момент_отключения', 'Дата окончания последней услуги Матч! Футбол/Весь Футбол_активна_на_момент_отключения',
'Дата окончания последней услуги Ultra_активна_на_момент_отключения', 'Дата окончания последней услуги Добавь Кино/Добавь Кино и 4К_активна_на_момент_отключения',
'Дата окончания последней услуги Матч Премьер_активна_на_момент_отключения', 'Дата окончания последней услуги START_активна_на_момент_отключения',
'Дата окончания последней услуги Premier_активна_на_момент_отключения', 'Дата окончания последней услуги Amediateka_активна_на_момент_отключения',
'Дата окончания последней услуги Мультирум_активна_на_момент_отключения']

Ottok = create_final_dataframe (Ottok, delete_columns)

In [42]:
Ottok.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4801498 entries, 0 to 4801497
Data columns (total 25 columns):
 #   Column                                    Dtype 
---  ------                                    ----- 
 0   Название откл тарифа                      object
 1   Название откл услуги                      object
 2   Длительность откл тарифа                  int64 
 3   Признак акционности откл тарифа           bool  
 4   Признак платности откл тарифа             bool  
 5   Признак кредита откл тарифа               bool  
 6   Признак активности откл тарифа            bool  
 7   Название спутника                         object
 8   Парк оборудования                         object
 9   Модель оборудования                       object
 10  Название организации дилеров              object
 11  Источник сигнала                          object
 12  Код типа послед ОУ                        object
 13  Признак активного кредита                 bool  
 14  Мобильный оператор

In [43]:
def final_get_column_sizes (df):
    results = []

    for column in df.columns:
        column_size = df[column].memory_usage (deep=True)
        column_size_kb = column_size / 1024

        dtype = df[column].dtype
        empty_strings = round (df[column].isnull().sum()/len (df[column]), 2)

        results.append ({
            'Название признака': column,
            'Тип данных': dtype,
            'Пустые значения': empty_strings
        })

    return results

final_results = final_get_column_sizes (Ottok)
for item in final_results:
    print (f'Название признака: {item['Название признака']:90} | Тип данных: {item['Тип данных']} | Пустые значения: {item['Пустые значения']} %')

Название признака: Название откл тарифа                                                                       | Тип данных: object | Пустые значения: 0.0 %
Название признака: Название откл услуги                                                                       | Тип данных: object | Пустые значения: 0.0 %
Название признака: Длительность откл тарифа                                                                   | Тип данных: int64 | Пустые значения: 0.0 %
Название признака: Признак акционности откл тарифа                                                            | Тип данных: bool | Пустые значения: 0.0 %
Название признака: Признак платности откл тарифа                                                              | Тип данных: bool | Пустые значения: 0.0 %
Название признака: Признак кредита откл тарифа                                                                | Тип данных: bool | Пустые значения: 0.0 %
Название признака: Признак активности откл тарифа                      

In [47]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
import itertools
from collections import defaultdict
warnings.filterwarnings('ignore')

def contains_separate_word_sled(feature_name):
    """
    Проверяет, содержит ли название признака отдельное слово 'след'
    """
    if not isinstance(feature_name, str):
        return False
    
    words = feature_name.lower().split()
    return 'след' in words

def period_in_range(week, year, start_year, end_year, start_week, end_week):
    """
    Проверяет, попадает ли период (неделя, год) в указанный диапазон
    """
    if year < start_year or year > end_year:
        return False
    
    # Для 2024 года: только недели 39-51
    if year == 2024:
        return start_week <= week <= end_week
    
    # Для 2025 года: только недели 39-51
    if year == 2025:
        return start_week <= week <= end_week
    
    return False

def get_zero_weeks_stats(proportion_table, count_table, start_year, end_year, start_week, end_week):
    """
    Для каждой категории каждого признака считает количество недель с нулём (не было абонентов)
    за выбранный диапазон годов и недель
    """
    print("Расчет статистики по неделям с нулями...")
    
    # Получаем все категории (исключая TOTAL_COUNT)
    category_data = proportion_table[proportion_table['value'] != 'TOTAL_COUNT'].copy()
    category_count_data = count_table[count_table['value'] != 'TOTAL_COUNT'].copy()
    
    # Получаем все периодические столбцы
    all_period_columns = [col for col in proportion_table.columns if col not in ['feature', 'value']]
    
    # Функция для сортировки периодов
    def period_sort_key(period):
        week, year = period.split('_')
        return (int(year), int(week))
    
    all_period_columns_sorted = sorted(all_period_columns, key=period_sort_key)
    
    # Фильтруем периоды по выбранному диапазону
    period_columns = []
    for period in all_period_columns_sorted:
        week, year = period.split('_')
        week, year = int(week), int(year)
        
        # Проверяем, попадает ли период в выбранный диапазон
        if period_in_range(week, year, start_year, end_year, start_week, end_week):
            period_columns.append(period)
    
    print(f"Анализируемый период для исключенных категорий: с н{start_week}/г{start_year} по н{end_week}/г{end_year}")
    print(f"Количество анализируемых недель: {len(period_columns)}")
    
    if len(period_columns) == 0:
        print("ОШИБКА: Нет данных в указанном диапазоне")
        return pd.DataFrame()
    
    results = []
    
    # Проходим по всем категориям
    for idx, (row, count_row) in enumerate(zip(category_data.iterrows(), category_count_data.iterrows())):
        row_data = row[1]
        count_data = count_row[1]
        
        feature = row_data['feature']
        value = row_data['value']
        
        # Исключаем признаки с отдельным словом 'след'
        if contains_separate_word_sled(feature):
            continue
        
        # Считаем количество недель с нулевым количеством абонентов
        zero_weeks_count = 0
        total_weeks_analyzed = 0
        
        for period in period_columns:
            count = count_data[period]
            if pd.isna(count) or count == 0:
                zero_weeks_count += 1
            total_weeks_analyzed += 1
        
        # Процент недель с нулями
        zero_percentage = (zero_weeks_count / total_weeks_analyzed * 100) if total_weeks_analyzed > 0 else 100
        
        # Определяем тип категории
        if zero_percentage > 85:
            category_type = 'excluded'  # Исключаем из анализа (>85%)
        elif zero_percentage > 30:  # >30% и <=85%
            category_type = 'type2'
        else:  # <=30%
            category_type = 'type1'
        
        results.append({
            'feature': feature,
            'value': value,
            'total_weeks': total_weeks_analyzed,
            'zero_weeks': zero_weeks_count,
            'zero_percentage': round(zero_percentage, 2),
            'category_type': category_type
        })
    
    zero_stats_df = pd.DataFrame(results)
    print(f"Проанализировано категорий для исключения: {len(zero_stats_df)}")
    
    return zero_stats_df

def calculate_category_statistics(proportion_table, count_table, zero_stats_df, start_year, end_year, start_week, end_week):
    """
    Рассчитывает статистику для категорий и классифицирует значения по неделям
    ВСЕ расчеты (статистика и классификация) проводятся ТОЛЬКО в указанном диапазоне
    """
    print("\nРасчет статистики и классификация категорий...")
    
    # Получаем все категории (исключая TOTAL_COUNT)
    category_data = proportion_table[proportion_table['value'] != 'TOTAL_COUNT'].copy()
    category_count_data = count_table[count_table['value'] != 'TOTAL_COUNT'].copy()
    
    # Получаем все периодические столбцы
    all_period_columns = [col for col in proportion_table.columns if col not in ['feature', 'value']]
    
    # Функция для сортировки периодов
    def period_sort_key(period):
        week, year = period.split('_')
        return (int(year), int(week))
    
    all_period_columns_sorted = sorted(all_period_columns, key=period_sort_key)
    
    # Фильтруем периоды по выбранному диапазону
    period_columns = []
    for period in all_period_columns_sorted:
        week, year = period.split('_')
        week, year = int(week), int(year)
        
        # Проверяем, попадает ли период в выбранный диапазон
        if period_in_range(week, year, start_year, end_year, start_week, end_week):
            period_columns.append(period)
    
    if len(period_columns) == 0:
        print("ОШИБКА: Нет данных в указанном диапазоне для расчета статистики")
        return pd.DataFrame()
    
    print(f"Период для расчета статистики и классификации: с н{start_week}/г{start_year} по н{end_week}/г{end_year}")
    print(f"Количество недель для анализа: {len(period_columns)}")
    
    results = []
    
    # Проходим по всем категориям
    for idx, (row, count_row) in enumerate(zip(category_data.iterrows(), category_count_data.iterrows())):
        row_data = row[1]
        count_data = count_row[1]
        
        feature = row_data['feature']
        value = row_data['value']
        
        # Исключаем признаки с отдельным словом 'след'
        if contains_separate_word_sled(feature):
            continue
        
        # Находим статистику для этой категории
        category_stats = zero_stats_df[
            (zero_stats_df['feature'] == feature) & 
            (zero_stats_df['value'] == value)
        ]
        
        if category_stats.empty:
            continue
            
        category_type = category_stats.iloc[0]['category_type']
        
        # Если категория исключена из анализа, пропускаем
        if category_type == 'excluded':
            continue
        
        # Собираем все значения за анализируемый период (только для расчета статистики)
        values = []
        for period in period_columns:
            count = count_data[period]
            if not pd.isna(count):
                values.append(count)
        
        if not values:
            continue
            
        # Преобразуем в numpy array для вычислений
        values_array = np.array(values)
        
        # Для категорий типа 1 (<=30% нулей): отсекаем 16.7% экстремумов
        if category_type == 'type1':
            # Сортируем значения
            sorted_values = np.sort(values_array)
            
            # Определяем индексы для отсечения 16.7% с каждого конца
            n = len(sorted_values)
            lower_cut_index = int(np.floor(0.167 * n))
            upper_cut_index = int(np.ceil(0.833 * n))
            
            # Отсекаем экстремумы
            trimmed_values = sorted_values[lower_cut_index:upper_cut_index]
            
            if len(trimmed_values) == 0:
                continue
                
            mean_value = np.mean(trimmed_values)
            std_value = np.std(trimmed_values)
        
        # Для категорий типа 2 (>30% и <=85% нулей): используем все недели с ненулевыми значениями
        else:  # category_type == 'type2'
            # Берем только ненулевые значения
            non_zero_values = values_array[values_array > 0]
            
            if len(non_zero_values) == 0:
                continue
                
            mean_value = np.mean(non_zero_values)
            std_value = np.std(non_zero_values)
        
        # Рассчитываем доверительный интервал (mean ± 2*std)
        lower_bound = mean_value - 2 * std_value
        upper_bound = mean_value + 2 * std_value
        
        # Корректируем нижнюю границу, чтобы она не была отрицательной
        lower_bound = max(0, lower_bound)
        
        # Теперь классифицируем значения ТОЛЬКО для недель в указанном диапазоне
        all_weeks_results = []
        
        for period in period_columns:
            week, year = period.split('_')
            week, year = int(week), int(year)
            
            # Получаем значение для этой недели
            count_value = count_data[period] if period in count_data else 0
            if pd.isna(count_value):
                count_value = 0
            
            # Определяем группу (исключаем "Нулевое значение")
            if count_value == 0:
                continue  # Пропускаем нулевые значения
            elif count_value < lower_bound:
                group = 'Ниже доверительного интервала'
            elif count_value > upper_bound:
                group = 'Выше доверительного интервала'
            else:
                group = 'В пределах доверительного интервала'
            
            all_weeks_results.append({
                'feature': feature,
                'value': value,
                'category_type': category_type,
                'week': week,
                'year': year,
                'period': f'н{week}/г{year}',
                'count': int(count_value),
                'group': group,
                'mean': round(mean_value, 2),
                'std': round(std_value, 2),
                'lower_bound': round(lower_bound, 2),
                'upper_bound': round(upper_bound, 2)
            })
        
        # Добавляем результаты для этой категории
        results.extend(all_weeks_results)
    
    classification_df = pd.DataFrame(results)
    print(f"Создана таблица классификации: {classification_df.shape}")
    
    # Проверяем диапазон дат в классификации
    if not classification_df.empty:
        years_in_classification = sorted(classification_df['year'].unique())
        print(f"Годы в классификации: {years_in_classification}")
        
        for year in years_in_classification:
            weeks_for_year = sorted(classification_df[classification_df['year'] == year]['week'].unique())
            print(f"  Год {year}: недели {weeks_for_year}")
            
            # Проверяем, что все недели в диапазоне 39-51
            invalid_weeks = [w for w in weeks_for_year if w < start_week or w > end_week]
            if invalid_weeks:
                print(f"  ВНИМАНИЕ: Найдены недели вне диапазона {start_week}-{end_week}: {invalid_weeks}")
            else:
                print(f"  ✓ Все недели в диапазоне {start_week}-{end_week}")
    
    return classification_df

def calculate_pair_statistics(proportion_table, count_table, start_year, end_year, start_week, end_week):
    """
    Рассчитывает статистику для пар категорий из разных признаков
    Совместные значения рассчитываются как СУММА значений двух категорий
    """
    print("\nРасчет статистики для пар категорий из разных признаков (сумма значений)...")
    
    # Получаем все категории (исключая TOTAL_COUNT)
    category_data = proportion_table[proportion_table['value'] != 'TOTAL_COUNT'].copy()
    category_count_data = count_table[count_table['value'] != 'TOTAL_COUNT'].copy()
    
    # Получаем все периодические столбцы
    all_period_columns = [col for col in proportion_table.columns if col not in ['feature', 'value']]
    
    # Функция для сортировки периодов
    def period_sort_key(period):
        week, year = period.split('_')
        return (int(year), int(week))
    
    all_period_columns_sorted = sorted(all_period_columns, key=period_sort_key)
    
    # Фильтруем периоды по выбранному диапазону
    period_columns = []
    for period in all_period_columns_sorted:
        week, year = period.split('_')
        week, year = int(week), int(year)
        
        # Проверяем, попадает ли период в выбранный диапазон
        if period_in_range(week, year, start_year, end_year, start_week, end_week):
            period_columns.append(period)
    
    if len(period_columns) == 0:
        print("ОШИБКА: Нет данных в указанном диапазоне для расчета статистики пар")
        return pd.DataFrame()
    
    print(f"Период для расчета статистики пар: с н{start_week}/г{start_year} по н{end_week}/г{end_year}")
    print(f"Количество недель для анализа пар: {len(period_columns)}")
    
    # Создаем словарь для быстрого доступа к данным по категориям
    print("Загрузка данных по категориям...")
    category_dict = {}
    total_categories = len(category_data)
    
    for idx, (row, count_row) in enumerate(zip(category_data.iterrows(), category_count_data.iterrows())):
        if (idx + 1) % 500 == 0:
            print(f"  Загружено категорий: {idx + 1}/{total_categories}")
        
        row_data = row[1]
        count_data = count_row[1]
        
        feature = row_data['feature']
        value = row_data['value']
        
        # Исключаем признаки с отдельным словом 'след'
        if contains_separate_word_sled(feature):
            continue
        
        # Сохраняем данные по категории
        category_dict[(feature, value)] = {
            'counts': {period: (count_data[period] if period in count_data and not pd.isna(count_data[period]) else 0) 
                      for period in period_columns},
            'feature': feature,
            'value': value
        }
    
    print(f"Загружено категорий для анализа пар: {len(category_dict)}")
    
    # Группируем категории по признакам
    print("Группировка категорий по признакам...")
    categories_by_feature = {}
    for (feature, value), data in category_dict.items():
        if feature not in categories_by_feature:
            categories_by_feature[feature] = []
        categories_by_feature[feature].append(value)
    
    # Получаем список всех признаков
    all_features = list(categories_by_feature.keys())
    print(f"Всего признаков для анализа пар: {len(all_features)}")
    
    # Выводим статистику по количеству категорий на признак
    categories_per_feature = [len(categories_by_feature[feature]) for feature in all_features]
    if categories_per_feature:
        print(f"Среднее количество категорий на признак: {np.mean(categories_per_feature):.1f}")
        print(f"Максимальное количество категорий на признак: {max(categories_per_feature)}")
        print(f"Минимальное количество категорий на признак: {min(categories_per_feature)}")
    
    # Предварительная фильтрация: исключаем признаки с очень большим количеством категорий
    # для предотвращения комбинаторного взрыва
    max_categories_per_feature = 50  # Максимальное количество категорий на признак для анализа
    filtered_features = []
    
    for feature in all_features:
        if len(categories_by_feature[feature]) <= max_categories_per_feature:
            filtered_features.append(feature)
        else:
            print(f"  Признак '{feature}' пропущен (слишком много категорий: {len(categories_by_feature[feature])})")
    
    print(f"Признаков после фильтрации: {len(filtered_features)} (исключено: {len(all_features) - len(filtered_features)})")
    
    if len(filtered_features) < 2:
        print("Недостаточно признаков для анализа пар")
        return pd.DataFrame()
    
    results = []
    total_pairs_analyzed = 0
    
    print("\nАнализ пар категорий...")
    # Проходим по всем парам признаков
    for i in range(len(filtered_features)):
        feature1 = filtered_features[i]
        
        for j in range(i + 1, len(filtered_features)):
            feature2 = filtered_features[j]
            
            # Получаем категории для каждого признака
            categories1 = categories_by_feature[feature1]
            categories2 = categories_by_feature[feature2]
            
            # Проходим по всем парам категорий
            for cat1 in categories1:
                key1 = (feature1, cat1)
                if key1 not in category_dict:
                    continue
                    
                data1 = category_dict[key1]
                
                for cat2 in categories2:
                    key2 = (feature2, cat2)
                    if key2 not in category_dict:
                        continue
                    
                    data2 = category_dict[key2]
                    total_pairs_analyzed += 1
                    
                    if total_pairs_analyzed % 1000 == 0:
                        print(f"  Проанализировано пар: {total_pairs_analyzed}")
                    
                    # Рассчитываем совместные значения как СУММУ
                    joint_values = []
                    
                    for period in period_columns:
                        count1 = data1['counts'].get(period, 0)
                        count2 = data2['counts'].get(period, 0)
                        
                        # СУММА значений двух категорий
                        joint_value = count1 + count2
                        
                        joint_values.append(joint_value)
                    
                    if not joint_values:
                        continue
                    
                    # Преобразуем в numpy array для вычислений
                    values_array = np.array(joint_values)
                    
                    # Определяем тип пары на основе количества нулевых значений
                    non_zero_values = values_array[values_array > 0]
                    zero_percentage = (len(values_array) - len(non_zero_values)) / len(values_array) * 100
                    
                    # Если пара категорий имеет >85% нулевых недель - пропускаем
                    if zero_percentage > 85:
                        continue
                    
                    # Определяем тип пары
                    if zero_percentage > 30:
                        category_type = 'type2'
                    else:
                        category_type = 'type1'
                    
                    # Для пар типа 1 (<=30% нулей): отсекаем 16.7% экстремумов
                    if category_type == 'type1':
                        # Сортируем значения
                        sorted_values = np.sort(values_array)
                        
                        # Определяем индексы для отсечения 16.7% с каждого конца
                        n = len(sorted_values)
                        lower_cut_index = int(np.floor(0.167 * n))
                        upper_cut_index = int(np.ceil(0.833 * n))
                        
                        # Отсекаем экстремумы
                        trimmed_values = sorted_values[lower_cut_index:upper_cut_index]
                        
                        if len(trimmed_values) == 0:
                            continue
                            
                        mean_value = np.mean(trimmed_values)
                        std_value = np.std(trimmed_values)
                    
                    # Для пар типа 2 (>30% и <=85% нулей): используем все недели с ненулевыми значениями
                    else:  # category_type == 'type2'
                        # Берем только ненулевые значения
                        non_zero_values = values_array[values_array > 0]
                        
                        if len(non_zero_values) == 0:
                            continue
                            
                        mean_value = np.mean(non_zero_values)
                        std_value = np.std(non_zero_values)
                    
                    # Рассчитываем доверительный интервал (mean ± 2*std)
                    lower_bound = mean_value - 2 * std_value
                    upper_bound = mean_value + 2 * std_value
                    
                    # Корректируем нижнюю границу, чтобы она не была отрицательной
                    lower_bound = max(0, lower_bound)
                    
                    # Теперь классифицируем значения ТОЛЬКО для недель в указанном диапазоне
                    for period_idx, period in enumerate(period_columns):
                        week, year = period.split('_')
                        week, year = int(week), int(year)
                        
                        # Получаем совместное значение для этой недели
                        joint_value = joint_values[period_idx]
                        
                        # Определяем группу (исключаем "Нулевое значение")
                        if joint_value == 0:
                            continue  # Пропускаем нулевые значения
                        elif joint_value < lower_bound:
                            group = 'Ниже доверительного интервала'
                        elif joint_value > upper_bound:
                            group = 'Выше доверительного интервала'
                        else:
                            group = 'В пределах доверительного интервала'
                        
                        # Получаем отдельные значения для каждой категории
                        count1 = data1['counts'].get(period, 0)
                        count2 = data2['counts'].get(period, 0)
                        
                        results.append({
                            'feature1': feature1,
                            'value1': cat1,
                            'feature2': feature2,
                            'value2': cat2,
                            'category_type': category_type,
                            'week': week,
                            'year': year,
                            'period': f'н{week}/г{year}',
                            'count1': int(count1),
                            'count2': int(count2),
                            'joint_count': int(joint_value),  # Сумма значений
                            'group': group,
                            'mean': round(mean_value, 2),
                            'std': round(std_value, 2),
                            'lower_bound': round(lower_bound, 2),
                            'upper_bound': round(upper_bound, 2)
                        })
    
    pair_classification_df = pd.DataFrame(results)
    print(f"\nСоздана таблица классификации пар: {pair_classification_df.shape}")
    print(f"Всего проанализировано пар: {total_pairs_analyzed}")
    
    if not pair_classification_df.empty:
        unique_pairs = pair_classification_df[['feature1', 'value1', 'feature2', 'value2']].drop_duplicates().shape[0]
        print(f"Уникальных пар категорий проанализировано: {unique_pairs}")
        
        # Статистика по типам пар
        if 'category_type' in pair_classification_df.columns:
            type_stats = pair_classification_df[['feature1', 'value1', 'feature2', 'value2', 'category_type']].drop_duplicates()['category_type'].value_counts()
            print("\nРаспределение пар по типам:")
            for cat_type, count in type_stats.items():
                type_name = "Тип 1 (≤30% нулей)" if cat_type == 'type1' else "Тип 2 (30-85% нулей)"
                print(f"  {type_name}: {count}")
        
        # Статистика по группам
        if 'group' in pair_classification_df.columns:
            group_stats = pair_classification_df['group'].value_counts()
            print("\nРаспределение пар по группам:")
            for group, count in group_stats.items():
                print(f"  {group}: {count}")
    
    return pair_classification_df

def find_category_pairs_above_interval(classification_df):
    """
    Находит пары категорий, которые одновременно попадают в группу 
    "Выше доверительного интервала" на одних и тех же неделях
    Возвращает таблицу с комбинациями пар категорий, количеством недель и списком недель
    """
    print("\nПоиск пар категорий выше доверительного интервала...")
    
    # Фильтруем только записи выше доверительного интервала
    above_interval_df = classification_df[classification_df['group'] == 'Выше доверительного интервала'].copy()
    
    if above_interval_df.empty:
        print("Нет записей выше доверительного интервала")
        return pd.DataFrame()
    
    # Создаем словарь для хранения пар категорий
    pairs_dict = {}
    
    # Группируем по неделям
    week_data = defaultdict(list)
    
    for _, row in above_interval_df.iterrows():
        key = (row['year'], row['week'])
        week_data[key].append({
            'feature': row['feature'],
            'value': row['value'],
            'count': row['count'],
            'period': f'н{row["week"]}/г{row["year"]}'
        })
    
    # Находим пары для каждой недели
    for (year, week), categories in week_data.items():
        if len(categories) < 2:
            continue
        
        # Создаем все возможные пары категорий
        for i in range(len(categories)):
            for j in range(i + 1, len(categories)):
                cat1 = categories[i]
                cat2 = categories[j]
                
                # Проверяем, что это разные признаки (не сравниваем категории одного признака)
                if cat1['feature'] != cat2['feature']:
                    # Создаем ключ для пары (упорядоченный)
                    if cat1['feature'] < cat2['feature']:
                        pair_key = (cat1['feature'], cat1['value'], cat2['feature'], cat2['value'])
                        feature1, value1, feature2, value2 = pair_key
                    else:
                        pair_key = (cat2['feature'], cat2['value'], cat1['feature'], cat1['value'])
                        feature2, value2, feature1, value1 = pair_key
                    
                    # Инициализируем запись для этой пары, если ее еще нет
                    if pair_key not in pairs_dict:
                        pairs_dict[pair_key] = {
                            'feature1': feature1,
                            'value1': value1,
                            'feature2': feature2,
                            'value2': value2,
                            'weeks_count': 0,
                            'weeks_list': [],
                            'total_count': 0
                        }
                    
                    # Обновляем информацию о паре
                    pairs_dict[pair_key]['weeks_count'] += 1
                    pairs_dict[pair_key]['weeks_list'].append(f'н{week}/г{year}')
                    pairs_dict[pair_key]['total_count'] += cat1['count'] + cat2['count']
    
    if pairs_dict:
        # Преобразуем словарь в список для DataFrame
        pairs_data = []
        for pair_key, pair_info in pairs_dict.items():
            # Сортируем список недель
            weeks_sorted = sorted(pair_info['weeks_list'])
            weeks_str = ', '.join(weeks_sorted)
            
            pairs_data.append({
                'feature1': pair_info['feature1'],
                'value1': pair_info['value1'],
                'feature2': pair_info['feature2'],
                'value2': pair_info['value2'],
                'weeks_count': pair_info['weeks_count'],
                'weeks_list': weeks_str,
                'total_count': pair_info['total_count']
            })
        
        pairs_df = pd.DataFrame(pairs_data)
        
        # Сортируем по количеству недель и общему количеству
        pairs_df = pairs_df.sort_values(['weeks_count', 'total_count'], ascending=[False, False])
        
        print(f"Найдено уникальных пар категорий: {len(pairs_df)}")
        
        # Статистика
        if len(pairs_df) > 0:
            print(f"Максимальное количество недель для пары: {pairs_df['weeks_count'].max()}")
            print(f"Среднее количество недель для пары: {pairs_df['weeks_count'].mean():.1f}")
        
        return pairs_df
    else:
        print("Не найдено пар категорий выше доверительного интервала")
        return pd.DataFrame()

def calculate_growth_contribution(count_table, start_year, end_year, start_week, end_week, prev_year, prev_week, prevprev_year, prevprev_week):
    """
    Рассчитывает вклад пар категорий разных признаков в общий прирост оттока
    между двумя неделями. Прирост = 100%, пары категорий дают совместный вклад.
    Ищет все комбинации пар категорий разных признаков.
    """
    print(f"\nРасчет вклада в прирост оттока...")
    print(f"Предыдущая неделя: н{prev_week}/г{prev_year}")
    print(f"Позапрошлая неделя: н{prevprev_week}/г{prevprev_year}")
    
    # Проверяем, что выбранные недели находятся в анализируемом диапазоне
    if not period_in_range(prev_week, prev_year, start_year, end_year, start_week, end_week):
        print(f"Предыдущая неделя не входит в анализируемый диапазон")
        return pd.DataFrame()
    
    if not period_in_range(prevprev_week, prevprev_year, start_year, end_year, start_week, end_week):
        print(f"Позапрошлая неделя не входит в анализируемый диапазон")
        return pd.DataFrame()
    
    # Получаем столбцы для выбранных недель
    prev_period = f"{prev_week}_{prev_year}"
    prevprev_period = f"{prevprev_week}_{prevprev_year}"
    
    if prev_period not in count_table.columns or prevprev_period not in count_table.columns:
        print(f"ОШИБКА: Нет данных для выбранных недель")
        return pd.DataFrame()
    
    # Берем только категории (исключаем TOTAL_COUNT)
    category_data = count_table[count_table['value'] != 'TOTAL_COUNT'].copy()
    
    # Рассчитываем изменения для каждой категории
    category_changes = []
    
    for _, row in category_data.iterrows():
        prev_count = row[prev_period] if not pd.isna(row[prev_period]) else 0
        prevprev_count = row[prevprev_period] if not pd.isna(row[prevprev_period]) else 0
        
        change = prev_count - prevprev_count
        
        if change > 0:  # Только положительные изменения (прирост)
            category_changes.append({
                'feature': row['feature'],
                'value': row['value'],
                'prev_count': int(prev_count),
                'prevprev_count': int(prevprev_count),
                'change': int(change)
            })
    
    if not category_changes:
        print("Нет положительных изменений между выбранными неделями")
        return pd.DataFrame()
    
    # Создаем DataFrame с изменениями
    changes_df = pd.DataFrame(category_changes)
    
    # Суммируем общий прирост по всем категориям
    total_growth = changes_df['change'].sum()
    
    if total_growth == 0:
        print("Общий прирост равен 0")
        return pd.DataFrame()
    
    print(f"Общий прирост оттока: {total_growth} (100%)")
    
    # Рассчитываем вклад каждой категории в процентах
    changes_df['contribution_percent'] = (changes_df['change'] / total_growth * 100)
    
    # Сортируем по вкладу
    changes_df = changes_df.sort_values('contribution_percent', ascending=False)
    
    print(f"\nТоп-10 категорий по вкладу в прирост:")
    for idx, row in changes_df.head(10).iterrows():
        print(f"  {row['feature']}:{row['value']} - {row['contribution_percent']:.2f}% ({row['change']})")
    
    # Группируем категории по признакам
    features_data = {}
    for feature, group in changes_df.groupby('feature'):
        # Берем все категории с положительным вкладом
        features_data[feature] = group.sort_values('contribution_percent', ascending=False)
    
    # Находим ВСЕ комбинации пар категорий из РАЗНЫХ признаков
    pairs_contribution = []
    feature_list = list(features_data.keys())
    
    print(f"\nПоиск комбинаций пар категорий разных признаков...")
    print(f"Количество признаков с приростом: {len(feature_list)}")
    
    # Для каждого признака берем все категории с вкладом > 0.1%
    for i in range(len(feature_list)):
        for j in range(i + 1, len(feature_list)):
            feature1 = feature_list[i]
            feature2 = feature_list[j]
            
            # Берем все категории из каждого признака
            cats1 = features_data[feature1]
            cats2 = features_data[feature2]
            
            # Создаем все возможные пары
            for _, cat1 in cats1.iterrows():
                for _, cat2 in cats2.iterrows():
                    joint_contribution = cat1['contribution_percent'] + cat2['contribution_percent']
                    
                    pairs_contribution.append({
                        'feature1': feature1,
                        'value1': cat1['value'],
                        'feature2': feature2,
                        'value2': cat2['value'],
                        'joint_contribution': round(joint_contribution, 2)
                    })
    
    if pairs_contribution:
        pairs_contrib_df = pd.DataFrame(pairs_contribution)
        
        # Сортируем по совместному вкладу
        pairs_contrib_df = pairs_contrib_df.sort_values('joint_contribution', ascending=False)
        
        print(f"\nНайдено всех комбинаций пар категорий: {len(pairs_contrib_df)}")
        
        # Берем топ-30 пар с наибольшим совместным вкладом
        selected_pairs = pairs_contrib_df.head(30).copy()
        
        print(f"\nТоп-{len(selected_pairs)} пар категорий по совместному вкладу:")
        print(f"Суммарный вклад: {selected_pairs['joint_contribution'].sum():.1f}%")
        
        for idx, row in selected_pairs.head(10).iterrows():
            print(f"  {idx+1}. {row['feature1']}:{row['value1']} + {row['feature2']}:{row['value2']} = {row['joint_contribution']}%")
        
        if len(selected_pairs) > 10:
            print(f"  ... и еще {len(selected_pairs) - 10} пар")
        
        return selected_pairs
    else:
        print("Не удалось найти пары категорий с совместным вкладом")
        return pd.DataFrame()

def create_weekly_proportion_table(df, week_column, year_column, start_year, end_year, start_week, end_week):
    """
    Создаёт таблицу пропорций по неделям и годам для каждого признака
    ТОЛЬКО для указанного диапазона дат
    """
    print(f"\nСоздание таблиц пропорций и количеств...")
    print(f"Диапазон данных: с н{start_week}/г{start_year} по н{end_week}/г{end_year}")
    
    # Создаем список допустимых периодов СТРОГО по указанному диапазону
    valid_periods = []
    
    # Для 2024 года: недели с start_week по end_week
    if start_year == 2024:
        for week in range(start_week, end_week + 1):
            valid_periods.append(f"{week}_{start_year}")
    
    # Для 2025 года: недели с start_week по end_week
    if end_year == 2025:
        for week in range(start_week, end_week + 1):
            valid_periods.append(f"{week}_{end_year}")
    
    # Преобразуем в набор для быстрого поиска
    valid_periods_set = set(valid_periods)
    
    # Создаем столбец с периодом для фильтрации
    df_filtered = df.copy()
    df_filtered['period'] = df_filtered[week_column].astype(str) + '_' + df_filtered[year_column].astype(str)
    
    # Фильтруем только допустимые периоды
    df_filtered = df_filtered[df_filtered['period'].isin(valid_periods_set)].copy()
    
    if df_filtered.empty:
        print("ОШИБКА: Нет данных в указанном диапазоне")
        return pd.DataFrame(), pd.DataFrame()
    
    print(f"Данные после фильтрации: {df_filtered.shape}")
    
    # Проверяем какие годы и недели попали
    years_in_data = sorted(df_filtered[year_column].unique())
    print(f"Годы в данных: {years_in_data}")
    
    for year in years_in_data:
        weeks_for_year = sorted(df_filtered[df_filtered[year_column] == year][week_column].unique())
        print(f"  Год {year}: недели {weeks_for_year}")
    
    # список всех периодов в отфильтрованных данных
    all_periods = sorted(df_filtered['period'].unique(), 
                         key=lambda x: (int(x.split('_')[1]), int(x.split('_')[0])))  # Сортировка по году, затем по неделе
    
    # все нечисловые колонки (кроме недельной и годовой)
    non_numeric_columns = df_filtered.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    
    # Удаляем служебные колонки
    columns_to_remove = [week_column, year_column, 'period']
    for col in columns_to_remove:
        if col in non_numeric_columns:
            non_numeric_columns.remove(col)
    
    results = []
    count_results = []  # абсолютные количества
    
    # по периодам
    total_counts = df_filtered['period'].value_counts().sort_index()
    total_row = {'feature': 'TOTAL_COUNT', 'value': 'TOTAL_COUNT'}
    total_count_row = {'feature': 'TOTAL_COUNT', 'value': 'TOTAL_COUNT'}
    for period in all_periods:
        total_row[period] = total_counts.get(period, 0)
        total_count_row[period] = total_counts.get(period, 0)
    results.append(total_row)
    count_results.append(total_count_row)
    
    print(f"Обрабатываем признаки: {len(non_numeric_columns)} признаков")
    print(f"Периоды для анализа: {len(all_periods)} периодов")
    
    # Для каждого категориального признака
    for feature_idx, feature in enumerate(non_numeric_columns):
        if (feature_idx + 1) % 10 == 0:
            print(f"Обработан признаков: {feature_idx + 1}/{len(non_numeric_columns)}")
        
        # Для булевых признаков - в строки
        if df_filtered[feature].dtype == 'bool':
            temp_df = df_filtered.copy()
            temp_df[feature] = temp_df[feature].astype(str)
        else:
            temp_df = df_filtered
        
        # подсчёт кросс-таблицы: периоды vs значения признака
        cross_tab = pd.crosstab(temp_df['period'], temp_df[feature])
        
        # Преобразование в пропорции по периодам (доли внутри признака)
        cross_tab_prop = cross_tab.div(cross_tab.sum(axis=1), axis=0)
        
        # Для каждого значения признака
        for value in cross_tab_prop.columns:
            row_data = {'feature': feature, 'value': str(value)}
            count_row_data = {'feature': feature, 'value': str(value)}
            
            # Пропорции и количества по периодам
            for period in all_periods:
                if period in cross_tab_prop.index:
                    row_data[period] = cross_tab_prop.loc[period, value]
                    count_row_data[period] = cross_tab.loc[period, value] if period in cross_tab.index else 0
                else:
                    row_data[period] = 0
                    count_row_data[period] = 0
            
            results.append(row_data)
            count_results.append(count_row_data)
    
    # итоговые таблицы
    proportion_table = pd.DataFrame(results)
    count_table = pd.DataFrame(count_results)
    
    # Заполнение NaN нулями
    period_cols = [col for col in proportion_table.columns if col not in ['feature', 'value']]
    proportion_table[period_cols] = proportion_table[period_cols].fillna(0)
    count_table[period_cols] = count_table[period_cols].fillna(0)
    
    print(f"Создана таблица пропорций с {len(proportion_table)} строками")
    print(f"Создана таблица количеств с {len(count_table)} строкой")
    
    # Проверка диапазона в итоговых таблицах
    if period_cols:
        periods_info = []
        for period in period_cols[:5] + period_cols[-5:]:
            week, year = period.split('_')
            periods_info.append(f"н{week}/г{year}")
        
        print(f"\nПроверка диапазона в таблицах:")
        print(f"  Всего периодов: {len(period_cols)}")
        print(f"  Примеры периодов: {periods_info}")
    
    return proportion_table, count_table

def save_results_to_excel(excluded_categories_df, classification_df, pair_classification_df, pairs_df, growth_pairs_df, 
                         start_year, end_year, start_week, end_week,
                         filename='Аномалии по неделям СКО.xlsx'):
    """
    Сохраняет результаты анализа в Excel файл с несколькими листами
    """
    print(f"\nСохранение результатов в файл {filename}...")
    
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        # Лист 1: Исключенные категории (с нулями > 85%)
        if not excluded_categories_df.empty:
            excluded_formatted = excluded_categories_df[[
                'feature', 'value', 'total_weeks', 'zero_weeks', 'zero_percentage'
            ]].copy()
            excluded_formatted = excluded_formatted.rename(columns={
                'feature': 'Признак',
                'value': 'Категория',
                'total_weeks': 'Всего недель в анализе',
                'zero_weeks': 'Недель с нулями',
                'zero_percentage': 'Процент недель с нулями'
            })
            excluded_formatted = excluded_formatted.sort_values('Процент недель с нулями', ascending=False)
            excluded_formatted.to_excel(writer, sheet_name='Исключенные категории', index=False)
            
            worksheet = writer.sheets['Исключенные категории']
            worksheet.column_dimensions['A'].width = 30
            worksheet.column_dimensions['B'].width = 30
            worksheet.column_dimensions['C'].width = 20
            worksheet.column_dimensions['D'].width = 20
            worksheet.column_dimensions['E'].width = 25
            
            print(f"Сохранено исключенных категорий: {len(excluded_categories_df)}")
        else:
            pd.DataFrame({'Сообщение': ['Нет категорий для исключения']}).to_excel(
                writer, sheet_name='Исключенные категории', index=False
            )
        
        # Лист 2: Классификация по неделям (без нулевых значений)
        if not classification_df.empty:
            classification_formatted = classification_df[[
                'feature', 'value', 'category_type', 'week', 'year', 'period',
                'count', 'group', 'mean', 'std', 'lower_bound', 'upper_bound'
            ]].copy()
            
            type_translation = {
                'type1': 'Тип 1 (≤30% нулей)',
                'type2': 'Тип 2 (30-85% нулей)'
            }
            classification_formatted['category_type'] = classification_formatted['category_type'].map(type_translation)
            
            classification_formatted = classification_formatted.rename(columns={
                'feature': 'Признак',
                'value': 'Категория',
                'category_type': 'Тип категории',
                'week': 'Номер недели',
                'year': 'Год',
                'period': 'Период',
                'count': 'Количество на неделе',
                'group': 'Группа',
                'mean': 'Среднее значение',
                'std': 'Стандартное отклонение',
                'lower_bound': 'Нижняя граница интервала',
                'upper_bound': 'Верхняя граница интервала'
            })
            
            classification_formatted = classification_formatted.sort_values(['Признак', 'Категория', 'Год', 'Номер недели'])
            classification_formatted.to_excel(writer, sheet_name='Классификация по неделям', index=False)
            
            worksheet = writer.sheets['Классификация по неделям']
            for col in ['A', 'B', 'G']:
                worksheet.column_dimensions[col].width = 30
            for col in ['C', 'D', 'E', 'F']:
                worksheet.column_dimensions[col].width = 15
            for col in ['H', 'I', 'J', 'K', 'L']:
                worksheet.column_dimensions[col].width = 20
            
            print(f"Сохранено записей классификации: {len(classification_df)}")
        else:
            pd.DataFrame({'Сообщение': ['Нет данных для классификации']}).to_excel(
                writer, sheet_name='Классификация по неделям', index=False
            )
        
        # Лист 3: Классификация пар категорий
        if not pair_classification_df.empty:
            pair_classification_formatted = pair_classification_df[[
                'feature1', 'value1', 'feature2', 'value2', 'category_type', 
                'week', 'year', 'period', 'count1', 'count2', 'joint_count',
                'group', 'mean', 'std', 'lower_bound', 'upper_bound'
            ]].copy()
            
            type_translation = {
                'type1': 'Тип 1 (≤30% нулей)',
                'type2': 'Тип 2 (30-85% нулей)'
            }
            pair_classification_formatted['category_type'] = pair_classification_formatted['category_type'].map(type_translation)
            
            pair_classification_formatted = pair_classification_formatted.rename(columns={
                'feature1': 'Признак 1',
                'value1': 'Категория 1',
                'feature2': 'Признак 2',
                'value2': 'Категория 2',
                'category_type': 'Тип пары',
                'week': 'Номер недели',
                'year': 'Год',
                'period': 'Период',
                'count1': 'Количество 1',
                'count2': 'Количество 2',
                'joint_count': 'Совместное количество (сумма)',
                'group': 'Группа',
                'mean': 'Среднее значение суммы',
                'std': 'Стандартное отклонение суммы',
                'lower_bound': 'Нижняя граница интервала',
                'upper_bound': 'Верхняя граница интервала'
            })
            
            pair_classification_formatted = pair_classification_formatted.sort_values(
                ['Признак 1', 'Категория 1', 'Признак 2', 'Категория 2', 'Год', 'Номер недели']
            )
            pair_classification_formatted.to_excel(writer, sheet_name='Классификация пар', index=False)
            
            worksheet = writer.sheets['Классификация пар']
            worksheet.column_dimensions['A'].width = 25  # Признак 1
            worksheet.column_dimensions['B'].width = 30  # Категория 1
            worksheet.column_dimensions['C'].width = 25  # Признак 2
            worksheet.column_dimensions['D'].width = 30  # Категория 2
            worksheet.column_dimensions['E'].width = 15  # Тип пары
            worksheet.column_dimensions['F'].width = 15  # Неделя
            worksheet.column_dimensions['G'].width = 15  # Год
            worksheet.column_dimensions['H'].width = 15  # Период
            worksheet.column_dimensions['I'].width = 15  # Количество 1
            worksheet.column_dimensions['J'].width = 15  # Количество 2
            worksheet.column_dimensions['K'].width = 25  # Совместное количество (сумма)
            worksheet.column_dimensions['L'].width = 25  # Группа
            worksheet.column_dimensions['M'].width = 20  # Среднее
            worksheet.column_dimensions['N'].width = 20  # Стандартное отклонение
            worksheet.column_dimensions['O'].width = 20  # Нижняя граница
            worksheet.column_dimensions['P'].width = 20  # Верхняя граница
            
            print(f"Сохранено записей классификации пар: {len(pair_classification_df)}")
        else:
            pd.DataFrame({'Сообщение': ['Нет данных для классификации пар']}).to_excel(
                writer, sheet_name='Классификация пар', index=False
            )
        
        # Лист 4: Пары категорий выше доверительного интервала
        if not pairs_df.empty:
            pairs_formatted = pairs_df.copy()
            pairs_formatted = pairs_formatted.rename(columns={
                'feature1': 'Признак 1',
                'value1': 'Категория 1',
                'feature2': 'Признак 2',
                'value2': 'Категория 2',
                'weeks_count': 'Количество недель',
                'weeks_list': 'Список недель',
                'total_count': 'Общее количество'
            })
            
            # Переупорядочиваем колонки
            pairs_formatted = pairs_formatted[[
                'Признак 1', 'Категория 1', 'Признак 2', 'Категория 2',
                'Количество недель', 'Список недель', 'Общее количество'
            ]]
            
            pairs_formatted.to_excel(writer, sheet_name='Пары выше интервала', index=False)
            
            worksheet = writer.sheets['Пары выше интервала']
            worksheet.column_dimensions['A'].width = 25  # Признак 1
            worksheet.column_dimensions['B'].width = 30  # Категория 1
            worksheet.column_dimensions['C'].width = 25  # Признак 2
            worksheet.column_dimensions['D'].width = 30  # Категория 2
            worksheet.column_dimensions['E'].width = 18  # Количество недель
            worksheet.column_dimensions['F'].width = 50  # Список недель
            worksheet.column_dimensions['G'].width = 18  # Общее количество
            
            print(f"Сохранено пар категорий выше интервала: {len(pairs_df)}")
        else:
            pd.DataFrame({'Сообщение': ['Нет пар категорий выше доверительного интервала']}).to_excel(
                writer, sheet_name='Пары выше интервала', index=False
            )
        
        # Лист 5: Вклад пар категорий в прирост оттока
        if not growth_pairs_df.empty:
            growth_pairs_formatted = growth_pairs_df.copy()
            growth_pairs_formatted = growth_pairs_formatted.rename(columns={
                'feature1': 'Признак 1',
                'value1': 'Категория 1',
                'feature2': 'Признак 2',
                'value2': 'Категория 2',
                'joint_contribution': 'Совместный вклад (%)'
            })
            
            # Переупорядочиваем колонки (только необходимые)
            growth_pairs_formatted = growth_pairs_formatted[[
                'Признак 1', 'Категория 1', 'Признак 2', 'Категория 2', 'Совместный вклад (%)'
            ]]
            
            growth_pairs_formatted.to_excel(writer, sheet_name='Вклад в прирост', index=False)
            
            worksheet = writer.sheets['Вклад в прирост']
            worksheet.column_dimensions['A'].width = 25  # Признак 1
            worksheet.column_dimensions['B'].width = 30  # Категория 1
            worksheet.column_dimensions['C'].width = 25  # Признак 2
            worksheet.column_dimensions['D'].width = 30  # Категория 2
            worksheet.column_dimensions['E'].width = 20  # Совместный вклад (%)
            
            # Суммарный вклад
            total_joint_contribution = growth_pairs_formatted['Совместный вклад (%)'].sum()
            print(f"Суммарный вклад сохраненных пар: {total_joint_contribution:.1f}%")
            print(f"Сохранено пар категорий по вкладу в прирост: {len(growth_pairs_df)}")
        else:
            pd.DataFrame({'Сообщение': ['Нет данных по вкладу в прирост']}).to_excel(
                writer, sheet_name='Вклад в прирост', index=False
            )
    
    print(f"Результаты успешно сохранены в файл: {filename}")

def main_analysis(df, week_column, year_column, 
                  start_year, end_year, start_week, end_week,
                  prev_year=None, prev_week=None, prevprev_year=None, prevprev_week=None):
    
    # Проверяем наличие необходимых колонок
    if week_column not in df.columns:
        print(f"ОШИБКА: Колонка '{week_column}' не найдена в данных")
        return None
    if year_column not in df.columns:
        print(f"ОШИБКА: Колонка '{year_column}' не найдена в данных")
        return None
    
    print("=" * 80)
    print(f"АНАЛИЗ ОТТОКА АБОНЕНТОВ")
    print("=" * 80)
    print(f"Размер исходных данных: {df.shape}")
    print(f"Диапазон анализа: с н{start_week}/г{start_year} по н{end_week}/г{end_year}")
    print(f"Конкретные периоды: н{start_week}-{end_week}/г{start_year} и н{start_week}-{end_week}/г{end_year}")
    
    if prev_year and prev_week and prevprev_year and prevprev_week:
        print(f"Сравнение недель: н{prev_week}/г{prev_year} vs н{prevprev_week}/г{prevprev_year}")
    
    # 1. Создаем таблицы пропорций и количеств ТОЛЬКО для указанного диапазона
    print("\n" + "-" * 80)
    print("1. Создание базовых таблиц (только для указанного диапазона)...")
    proportion_table, count_table = create_weekly_proportion_table(
        df, week_column, year_column, start_year, end_year, start_week, end_week
    )
    
    if proportion_table.empty or count_table.empty:
        print("ОШИБКА: Не удалось создать базовые таблицы")
        return None
    
    print(f"Таблица пропорций: {proportion_table.shape}")
    print(f"Таблица количеств: {count_table.shape}")
    
    # 2. Анализируем недели с нулями на выбранном диапазоне
    print("\n" + "-" * 80)
    print("2. Анализ недель с нулевыми значениями...")
    zero_stats_df = get_zero_weeks_stats(
        proportion_table, count_table, start_year, end_year, start_week, end_week
    )
    
    if zero_stats_df.empty:
        print("ОШИБКА: Не удалось проанализировать недели с нулями")
        return None
    
    # Разделяем на исключенные и анализируемые категории
    excluded_categories_df = zero_stats_df[zero_stats_df['category_type'] == 'excluded'].copy()
    analyzed_categories_df = zero_stats_df[zero_stats_df['category_type'] != 'excluded'].copy()
    
    print(f"Всего категорий: {len(zero_stats_df)}")
    print(f"Исключено категорий (нулей > 85%): {len(excluded_categories_df)}")
    print(f"Категорий для анализа: {len(analyzed_categories_df)}")
    
    # 3. Рассчитываем статистику и классифицируем категории
    #    ВСЕ расчеты ТОЛЬКО в указанном диапазоне
    print("\n" + "-" * 80)
    print("3. Расчет статистики и классификация категорий (только для указанного диапазона)...")
    classification_df = calculate_category_statistics(
        proportion_table, count_table, zero_stats_df, start_year, end_year, start_week, end_week
    )
    
    if classification_df.empty:
        print("ВНИМАНИЕ: Нет данных для классификации")
        classification_df = pd.DataFrame()
    
    # 4. Рассчитываем статистику для пар категорий из разных признаков (СУММА значений)
    print("\n" + "-" * 80)
    print("4. Расчет статистики для пар категорий из разных признаков (на основе суммы)...")
    pair_classification_df = calculate_pair_statistics(
        proportion_table, count_table, start_year, end_year, start_week, end_week
    )
    
    if pair_classification_df.empty:
        print("ВНИМАНИЕ: Нет данных для классификации пар")
        pair_classification_df = pd.DataFrame()
    
    # 5. Находим пары категорий выше доверительного интервала
    print("\n" + "-" * 80)
    print("5. Поиск пар категорий выше доверительного интервала...")
    pairs_df = pd.DataFrame()
    if not classification_df.empty:
        pairs_df = find_category_pairs_above_interval(classification_df)
    
    # 6. Рассчитываем вклад пар категорий в прирост оттока (если указаны недели для сравнения)
    growth_pairs_df = pd.DataFrame()
    if all(v is not None for v in [prev_year, prev_week, prevprev_year, prevprev_week]):
        print("\n" + "-" * 80)
        print("6. Расчет вклада в прирост оттока...")
        growth_pairs_df = calculate_growth_contribution(
            count_table, start_year, end_year, start_week, end_week,
            prev_year, prev_week, prevprev_year, prevprev_week
        )
    
    # 7. Сохраняем результаты
    print("\n" + "-" * 80)
    print("7. Сохранение результатов...")
    filename = f'Аномалии_н{start_week}-{end_week}_г{start_year}-{end_year}.xlsx'
    
    save_results_to_excel(
        excluded_categories_df, classification_df, pair_classification_df, pairs_df, growth_pairs_df,
        start_year, end_year, start_week, end_week, filename
    )
    
    # 8. Выводим итоговую статистику
    print("\n" + "=" * 80)
    print("ИТОГИ АНАЛИЗА")
    print("=" * 80)
    print(f"Диапазон анализа: н{start_week}/г{start_year} - н{end_week}/г{end_year}")
    print(f"Всего категорий обработано: {len(zero_stats_df)}")
    print(f"Категорий исключено (нулей > 85%): {len(excluded_categories_df)}")
    
    if not classification_df.empty:
        analyzed_categories_count = classification_df[['feature', 'value']].drop_duplicates().shape[0]
        print(f"Категорий проанализировано: {analyzed_categories_count}")
        
        # Статистика по типам категорий
        if 'category_type' in classification_df.columns:
            type_stats = classification_df[['feature', 'value', 'category_type']].drop_duplicates()['category_type'].value_counts()
            print("\nРаспределение по типам категорий:")
            for cat_type, count in type_stats.items():
                type_name = "Тип 1 (≤30% нулей)" if cat_type == 'type1' else "Тип 2 (30-85% нулей)"
                print(f"  {type_name}: {count}")
        
        # Статистика по неделям в классификации
        unique_weeks = classification_df[['week', 'year']].drop_duplicates().shape[0]
        print(f"\nНедель с данными в классификации: {unique_weeks}")
        
        # Статистика по группам
        if 'group' in classification_df.columns:
            group_stats = classification_df['group'].value_counts()
            print("\nРаспределение по группам в классификации:")
            for group, count in group_stats.items():
                print(f"  {group}: {count}")
    
    if not pair_classification_df.empty:
        analyzed_pairs_count = pair_classification_df[['feature1', 'value1', 'feature2', 'value2']].drop_duplicates().shape[0]
        print(f"\nПар категорий проанализировано: {analyzed_pairs_count}")
        
        # Статистика по типам пар
        if 'category_type' in pair_classification_df.columns:
            pair_type_stats = pair_classification_df[['feature1', 'value1', 'feature2', 'value2', 'category_type']].drop_duplicates()['category_type'].value_counts()
            print("\nРаспределение пар по типам:")
            for cat_type, count in pair_type_stats.items():
                type_name = "Тип 1 (≤30% нулей)" if cat_type == 'type1' else "Тип 2 (30-85% нулей)"
                print(f"  {type_name}: {count}")
        
        # Статистика по группам для пар
        if 'group' in pair_classification_df.columns:
            pair_group_stats = pair_classification_df['group'].value_counts()
            print("\nРаспределение пар по группам:")
            for group, count in pair_group_stats.items():
                print(f"  {group}: {count}")
    
    if not pairs_df.empty:
        print(f"\nПар категорий выше доверительного интервала: {len(pairs_df)}")
        print(f"Максимальное количество недель для одной пары: {pairs_df['weeks_count'].max()}")
    
    if not growth_pairs_df.empty:
        print(f"\nПар категорий по вкладу в прирост: {len(growth_pairs_df)}")
        total_contribution = growth_pairs_df['joint_contribution'].sum()
        print(f"Суммарный вклад выбранных пар: {total_contribution:.1f}% от общего прироста")
    
    print(f"\nРезультаты сохранены в файл: {filename}")
    print("=" * 80)
    
    return {
        'proportion_table': proportion_table,
        'count_table': count_table,
        'zero_stats_df': zero_stats_df,
        'excluded_categories_df': excluded_categories_df,
        'classification_df': classification_df,
        'pair_classification_df': pair_classification_df,
        'pairs_df': pairs_df,
        'growth_pairs_df': growth_pairs_df,
        'filename': filename
    }

# Настройки анализа
week_column = 'Дата отключения откл тарифа_Номер недели'
year_column = 'Дата отключения откл тарифа_Год'

# Диапазон для расчета статистики (ПРИМЕР: с 39 недели 2024 по 51 неделю 2025)
start_year = 2024  # начальный год
end_year = 2025    # конечный год
start_week = 39    # начальная неделя
end_week = 51      # конечная неделя

# Недели для сравнения прироста (опционально)
prev_year = 2025      # год предыдущей недели
prev_week = 51        # номер предыдущей недели
prevprev_year = 2025  # год позапрошлой недели
prevprev_week = 50    # номер позапрошлой недели

# Проверка данных
print(f"Тип данных Ottok: {type(Ottok)}")
print(f"Размер Ottok: {Ottok.shape}")
print(f"Колонки в Ottok: {list(Ottok.columns)[:10]}...")

if week_column not in Ottok.columns:
    print(f"ОШИБКА: Колонка '{week_column}' не найдена в Ottok!")
    print(f"Доступные колонки: {list(Ottok.columns)}")
elif year_column not in Ottok.columns:
    print(f"ОШИБКА: Колонка '{year_column}' не найдена в Ottok!")
    print(f"Доступные колонки: {list(Ottok.columns)}")
else:
    # Запуск анализа
    results = main_analysis(
        Ottok, week_column, year_column,
        start_year, end_year, start_week, end_week,
        prev_year, prev_week, prevprev_year, prevprev_week
    )
    
    if results:
        print("\nАнализ успешно завершен!")
        
        print("\nКраткий обзор результатов:")
        print(f"1. Исключено категорий: {len(results['excluded_categories_df'])}")
        
        if not results['classification_df'].empty:
            analyzed_categories = results['classification_df'][['feature', 'value']].drop_duplicates().shape[0]
            print(f"2. Проанализировано категорий: {analyzed_categories}")
            print(f"3. Записей в классификации: {len(results['classification_df'])}")
        
        if not results['pair_classification_df'].empty:
            analyzed_pairs = results['pair_classification_df'][['feature1', 'value1', 'feature2', 'value2']].drop_duplicates().shape[0]
            print(f"4. Проанализировано пар категорий: {analyzed_pairs}")
            print(f"5. Записей в классификации пар: {len(results['pair_classification_df'])}")
        
        if not results['pairs_df'].empty:
            print(f"6. Уникальных пар категорий выше интервала: {len(results['pairs_df'])}")
        
        if not results['growth_pairs_df'].empty:
            total_contrib = results['growth_pairs_df']['joint_contribution'].sum()
            print(f"7. Пар категорий по вкладу в прирост: {len(results['growth_pairs_df'])} (суммарно {total_contrib:.1f}%)")
            
        print(f"8. Файл с результатами: {results['filename']}")
        
        # Показать примеры исключенных категорий
        if not results['excluded_categories_df'].empty:
            print("\nПримеры исключенных категорий (первые 5):")
            display(results['excluded_categories_df'].head())
        
        # Показать примеры классификации
        if not results['classification_df'].empty:
            print("\nПримеры классификации (первые 5):")
            display(results['classification_df'].head())
        
        # Показать примеры классификации пар
        if not results['pair_classification_df'].empty:
            print("\nПримеры классификации пар (первые 5):")
            display(results['pair_classification_df'].head())
        
        # Показать примеры пар выше интервала
        if not results['pairs_df'].empty:
            print("\nПримеры пар категорий выше интервала (первые 5):")
            display(results['pairs_df'].head())
        
        # Показать примеры пар по вкладу в прирост
        if not results['growth_pairs_df'].empty:
            print("\nПримеры пар категорий по вкладу в прирост (первые 5):")
            display(results['growth_pairs_df'].head())
    else:
        print("\nАнализ завершен с ошибками.")

Тип данных Ottok: <class 'pandas.core.frame.DataFrame'>
Размер Ottok: (4801498, 25)
Колонки в Ottok: ['Название откл тарифа', 'Название откл услуги', 'Длительность откл тарифа', 'Признак акционности откл тарифа', 'Признак платности откл тарифа', 'Признак кредита откл тарифа', 'Признак активности откл тарифа', 'Название спутника', 'Парк оборудования', 'Модель оборудования']...
АНАЛИЗ ОТТОКА АБОНЕНТОВ
Размер исходных данных: (4801498, 25)
Диапазон анализа: с н39/г2024 по н51/г2025
Конкретные периоды: н39-51/г2024 и н39-51/г2025
Сравнение недель: н51/г2025 vs н50/г2025

--------------------------------------------------------------------------------
1. Создание базовых таблиц (только для указанного диапазона)...

Создание таблиц пропорций и количеств...
Диапазон данных: с н39/г2024 по н51/г2025
Данные после фильтрации: (1316821, 26)
Годы в данных: [2024, 2025]
  Год 2024: недели [39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
  Год 2025: недели [39, 40, 41, 42, 43, 44, 45, 46, 47, 48

,feature,value,total_weeks,zero_weeks,zero_percentage,category_type
0,Название откл тарифа,Единый Мульти Лайт 1 мес_П,26,25,96.15,excluded
2,Название откл тарифа,Единый 1080 10П,26,25,96.15,excluded
4,Название откл тарифа,Единый 1140 K,26,24,92.31,excluded
6,Название откл тарифа,Единый 1200 К,26,25,96.15,excluded
14,Название откл тарифа,Единый 1800_15m,26,24,92.31,excluded



Примеры классификации (первые 5):


,feature,value,category_type,week,year,period,count,group,mean,std,lower_bound,upper_bound
0,Название откл тарифа,Единый 1000_В,type1,39,2024,н39/г2024,8,Ниже доверительного интервала,548.89,102.68,343.53,754.24
1,Название откл тарифа,Единый 1000_В,type1,40,2024,н40/г2024,4,Ниже доверительного интервала,548.89,102.68,343.53,754.24
2,Название откл тарифа,Единый 1000_В,type1,41,2024,н41/г2024,53,Ниже доверительного интервала,548.89,102.68,343.53,754.24
3,Название откл тарифа,Единый 1000_В,type1,42,2024,н42/г2024,126,Ниже доверительного интервала,548.89,102.68,343.53,754.24
4,Название откл тарифа,Единый 1000_В,type1,43,2024,н43/г2024,321,Ниже доверительного интервала,548.89,102.68,343.53,754.24



Примеры классификации пар (первые 5):


,feature1,value1,feature2,value2,category_type,week,year,period,count1,count2,joint_count,group,mean,std,lower_bound,upper_bound
0,Название откл услуги,Доступ к сети Триколор ТВ,Признак акционности откл тарифа,False,type1,39,2024,н39/г2024,0,47792,47792,В пределах доверительного интервала,47728.06,1517.96,44692.14,50763.97
1,Название откл услуги,Доступ к сети Триколор ТВ,Признак акционности откл тарифа,False,type1,40,2024,н40/г2024,2,93376,93378,Выше доверительного интервала,47728.06,1517.96,44692.14,50763.97
2,Название откл услуги,Доступ к сети Триколор ТВ,Признак акционности откл тарифа,False,type1,41,2024,н41/г2024,0,47395,47395,В пределах доверительного интервала,47728.06,1517.96,44692.14,50763.97
3,Название откл услуги,Доступ к сети Триколор ТВ,Признак акционности откл тарифа,False,type1,42,2024,н42/г2024,1,48131,48132,В пределах доверительного интервала,47728.06,1517.96,44692.14,50763.97
4,Название откл услуги,Доступ к сети Триколор ТВ,Признак акционности откл тарифа,False,type1,43,2024,н43/г2024,0,43923,43923,Ниже доверительного интервала,47728.06,1517.96,44692.14,50763.97



Примеры пар категорий выше интервала (первые 5):


,feature1,value1,feature2,value2,weeks_count,weeks_list,total_count
14849,Парк оборудования,Mstar K2,Наименование региона,Кабардино-Балкарская Респ,6,"н40/г2024, н49/г2025, н50/г2024, н50/г2025, н5...",33985
1457,Признак кредита откл тарифа,False,Код типа послед ОУ,P,5,"н40/г2024, н47/г2024, н50/г2025, н51/г2024, н5...",624677
3505,Код типа послед ОУ,P,Обращался в КЦ,False,5,"н40/г2024, н47/г2024, н50/г2025, н51/г2024, н5...",613781
1482,Признак кредита откл тарифа,False,Обращался в КЦ,False,5,"н40/г2024, н47/г2024, н50/г2025, н51/г2024, н5...",612596
3503,Код типа послед ОУ,P,Указан номер телефона,True,5,"н40/г2024, н47/г2024, н50/г2025, н51/г2024, н5...",603156



Примеры пар категорий по вкладу в прирост (первые 5):


,feature1,value1,feature2,value2,joint_contribution
473,Вай-фай на оборудовании,False,Признак кредита откл тарифа,False,9.76
137633,Признак акционности откл тарифа,False,Признак кредита откл тарифа,False,9.65
6806,Код типа послед ОУ,P,Признак кредита откл тарифа,False,9.65
137523,Признак активности откл тарифа,True,Признак кредита откл тарифа,False,9.64
137412,Признак активного кредита,False,Признак кредита откл тарифа,False,9.64
